# 21. Identifier Landscape and Harmonization Strategy

## Objective

The objective of this notebook is to define a transparent and reproducible harmonization strategy for integrating cell line information across DepMap and GDSC.

Before constructing the final modeling cohort, it is necessary to establish how biological models will be identified, linked, and represented across datasets. Although the previous overlap analysis confirmed the existence of a large shared universe of cell lines, several identifier systems coexist within the available resources, including DepMap Model IDs (ACH), Sanger Model IDs (SIDM), cell line names, and alternative aliases.

This notebook focuses on characterizing the identifier landscape, evaluating potential ambiguities, and documenting the rules that will govern cross-dataset integration.

The analysis will address the following questions:

- Which identifier systems are available in each dataset?
- How unique and consistent are these identifiers?
- Are there ambiguous mappings between identifiers?
- How should duplicate or related models be handled?
- Which identifiers should be adopted as canonical references?
- Which biological annotations should be retained for downstream analyses?

The goal is not to construct the final modeling cohort, but rather to establish the harmonization framework that will enable reproducible integration in subsequent notebooks.

## Expected Outputs

By the end of this notebook, we aim to define:

- A canonical model identifier strategy.
- Cross-dataset mapping rules.
- Duplicate handling criteria.
- Biological annotation standards.
- A harmonized model reference table for downstream analyses.

The resulting harmonization decisions will serve as the foundation for cohort construction, pharmacogenomic integration, and all subsequent modeling steps.

In [1]:
# =============================================================================
# Imports
# =============================================================================

from pathlib import Path

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
# =============================================================================
# Paths
# =============================================================================

PROJECT_ROOT = Path.cwd().parents[1]

RAW_DATA_DIR = PROJECT_ROOT / "data" / "raw"
INTERIM_DATA_DIR = PROJECT_ROOT / "data" / "interim"

RAW_DATA_DIR, INTERIM_DATA_DIR

(WindowsPath('c:/Users/paula/OneDrive/Documentos/Proyectos/pancancer-epigenetics/data/raw'),
 WindowsPath('c:/Users/paula/OneDrive/Documentos/Proyectos/pancancer-epigenetics/data/interim'))

In [3]:
# =============================================================================
# Load harmonization seed table
# =============================================================================

shared_models_path = INTERIM_DATA_DIR / "20_shared_depmap_gdsc_models.csv"

shared_depmap_models = pd.read_csv(shared_models_path)

print(f"Shared DepMap-GDSC models: {shared_depmap_models.shape}")
display(shared_depmap_models.head())

Shared DepMap-GDSC models: (968, 8)


,ModelID,SangerModelID,COSMICID,CellLineName,OncotreeLineage,OncotreePrimaryDisease,OncotreeSubtype,CCLEName
0,ACH-000001,SIDM00105,905933.0,NIH:OVCAR-3,Ovary/Fallopian Tube,Ovarian Epithelial Tumor,High-Grade Serous Ovarian Cancer,NIHOVCAR3_OVARY
1,ACH-000002,SIDM00829,905938.0,HL-60,Myeloid,Acute Myeloid Leukemia,AML with Myelodysplasia-Related Changes,HL60_HAEMATOPOIETIC_AND_LYMPHOID_TISSUE
2,ACH-000004,SIDM00594,907053.0,HEL,Myeloid,Acute Myeloid Leukemia,"AML, NOS",HEL_HAEMATOPOIETIC_AND_LYMPHOID_TISSUE
3,ACH-000006,SIDM01023,908148.0,MONO-MAC-6,Myeloid,Acute Myeloid Leukemia,Acute Myeloid Leukemia,MONOMAC6_HAEMATOPOIETIC_AND_LYMPHOID_TISSUE
4,ACH-000007,SIDM00677,907795.0,LS513,Bowel,Colorectal Adenocarcinoma,Colon Adenocarcinoma,LS513_LARGE_INTESTINE


In [4]:
# =============================================================================
# Basic validation
# =============================================================================

assert shared_depmap_models.shape[0] > 0, "Shared model table is empty."
assert "ModelID" in shared_depmap_models.columns, "ModelID column is missing."
assert "SangerModelID" in shared_depmap_models.columns, "SangerModelID column is missing."

shared_depmap_models.columns.tolist()

['ModelID',
 'SangerModelID',
 'COSMICID',
 'CellLineName',
 'OncotreeLineage',
 'OncotreePrimaryDisease',
 'OncotreeSubtype',
 'CCLEName']

## Identifier Landscape

Before defining harmonization rules, we first evaluate the identifier systems available in the shared DepMap–GDSC model universe.

The objective is to understand:

- identifier completeness;
- identifier uniqueness;
- potential ambiguities across identifier systems;
- suitability of each identifier for downstream integration.

The available identifier systems are:

- ModelID (DepMap ACH identifier)
- SangerModelID (cross-dataset bridge)
- COSMICID
- CellLineName
- CCLEName

In [5]:
# =============================================================================
# Identifier completeness
# =============================================================================

identifier_columns = [
    "ModelID",
    "SangerModelID",
    "COSMICID",
    "CellLineName",
    "CCLEName",
]

identifier_completeness = pd.DataFrame([
    {
        "identifier": col,
        "non_missing": shared_depmap_models[col].notna().sum(),
        "missing": shared_depmap_models[col].isna().sum(),
        "missing_pct": (
            shared_depmap_models[col].isna().mean() * 100
        ),
    }
    for col in identifier_columns
])

identifier_completeness

,identifier,non_missing,missing,missing_pct
0,ModelID,968,0,0.000000
1,SangerModelID,968,0,0.000000
2,COSMICID,946,22,2.272727
3,CellLineName,968,0,0.000000
4,CCLEName,968,0,0.000000


### Preliminary observations

All primary identifier systems except COSMICID are fully populated across the shared DepMap–GDSC model universe.

COSMICID presents a small proportion of missing values (~2.3%), making it unsuitable as a primary integration key.

At this stage, the strongest candidates for canonical model identification are:

- ModelID (DepMap ACH identifier)
- SangerModelID (cross-dataset bridge)

Further evaluation requires assessing identifier uniqueness and mapping cardinality.

In [6]:
# =============================================================================
# Identifier uniqueness
# =============================================================================

identifier_columns = [
    "ModelID",
    "SangerModelID",
    "COSMICID",
    "CellLineName",
    "CCLEName",
]

uniqueness_summary = pd.DataFrame([
    {
        "identifier": col,
        "n_rows": len(shared_depmap_models),
        "n_unique": shared_depmap_models[col].nunique(dropna=True),
        "duplicates": (
            len(shared_depmap_models)
            - shared_depmap_models[col].nunique(dropna=True)
        ),
    }
    for col in identifier_columns
])

uniqueness_summary

,identifier,n_rows,n_unique,duplicates
0,ModelID,968,968,0
1,SangerModelID,968,967,1
2,COSMICID,968,946,22
3,CellLineName,968,968,0
4,CCLEName,968,968,0


## SangerModelID Ambiguity Analysis

A single Sanger model identifier appears more than once within the shared model universe.

Because SangerModelID is the primary bridge between DepMap and GDSC, this ambiguity must be characterized before defining canonical integration rules.

In [7]:
# =============================================================================
# Duplicated SangerModelIDs
# =============================================================================

duplicated_sanger_ids = (
    shared_depmap_models["SangerModelID"]
    .value_counts()
    .loc[lambda x: x > 1]
)

duplicated_sanger_ids

SangerModelID
SIDM00117    2
Name: count, dtype: int64

In [8]:
print(shared_depmap_models.loc[
    shared_depmap_models["SangerModelID"].isin(
        duplicated_sanger_ids.index
    )
].sort_values("SangerModelID"))

        ModelID SangerModelID  COSMICID CellLineName OncotreeLineage  \
510  ACH-000837     SIDM00117       NaN     NCI-H322            Lung   
832  ACH-002172     SIDM00117  905967.0    NCI-H322M            Lung   

         OncotreePrimaryDisease      OncotreeSubtype       CCLEName  
510  Non-Small Cell Lung Cancer  Lung Adenocarcinoma   NCIH322_LUNG  
832  Non-Small Cell Lung Cancer  Lung Adenocarcinoma  NCIH322M_LUNG  


### Observation

A single SangerModelID (SIDM00117) maps to two distinct DepMap ModelIDs.

This indicates that SangerModelID is not globally unique within the shared model universe.

Consequently, SangerModelID cannot be considered a canonical model identifier, although it remains the primary cross-dataset bridge between DepMap and GDSC.

In [9]:
# =============================================================================
# ACH -> SIDM cardinality
# =============================================================================

ach_to_sidm = (
    shared_depmap_models
    .groupby("ModelID")["SangerModelID"]
    .nunique()
)

ach_to_sidm.value_counts().sort_index()

SangerModelID
1    968
Name: count, dtype: int64

In [10]:
# =============================================================================
# SIDM -> ACH cardinality
# =============================================================================

sidm_to_ach = (
    shared_depmap_models
    .groupby("SangerModelID")["ModelID"]
    .nunique()
)

sidm_to_ach.value_counts().sort_index()

ModelID
1    966
2      1
Name: count, dtype: int64

### Cardinality assessment

The mapping between DepMap ModelID (ACH) and SangerModelID (SIDM) is almost entirely one-to-one.

Results show:

- 968/968 ACH identifiers map to exactly one SIDM.
- 966/967 SIDM identifiers map to exactly one ACH.
- A single SIDM (SIDM00117) maps to two distinct ACH identifiers.

Therefore:

- ModelID behaves as a globally unique model identifier.
- SangerModelID behaves as a near-unique integration identifier but is not strictly one-to-one.
- ModelID is retained as the canonical model identifier for downstream analyses.
- SangerModelID is retained as the primary cross-dataset bridge to GDSC.

In [11]:
annotation_columns = [
    "OncotreeLineage",
    "OncotreePrimaryDisease",
    "OncotreeSubtype"
]

shared_depmap_models[annotation_columns].nunique()

OncotreeLineage            28
OncotreePrimaryDisease     75
OncotreeSubtype           146
dtype: int64

In [12]:
shared_depmap_models[annotation_columns].isna().sum()


OncotreeLineage           0
OncotreePrimaryDisease    0
OncotreeSubtype           0
dtype: int64

## Biological Annotation Assessment

Three biological annotation levels are available across the entire shared model universe:

- OncotreeLineage
- OncotreePrimaryDisease
- OncotreeSubtype

All annotation systems are fully populated with no missing values.

The annotation hierarchy provides increasing biological granularity:

- OncotreeLineage: broad tissue-level classification.
- OncotreePrimaryDisease: disease-level classification.
- OncotreeSubtype: fine-grained disease annotation.

Given their complete coverage and hierarchical structure, these annotations are retained for downstream analyses.

For most cross-cancer analyses, OncotreeLineage will serve as the primary biological stratification variable, while disease and subtype annotations will be preserved for higher-resolution analyses and sensitivity assessments.

In [13]:
harmonization_decisions = pd.DataFrame([
    {
        "component": "Canonical model identifier",
        "selected_variable": "ModelID",
        "rationale": "Globally unique and fully populated",
    },
    {
        "component": "Cross-dataset bridge",
        "selected_variable": "SangerModelID",
        "rationale": "Primary identifier shared between DepMap and GDSC",
    },
    {
        "component": "Primary lineage annotation",
        "selected_variable": "OncotreeLineage",
        "rationale": "Fully populated and suitable for lineage-aware analyses",
    },
    {
        "component": "Primary disease annotation",
        "selected_variable": "OncotreePrimaryDisease",
        "rationale": "Fully populated disease-level annotation",
    },
    {
        "component": "Subtype annotation",
        "selected_variable": "OncotreeSubtype",
        "rationale": "Highest available biological granularity",
    },
])

In [14]:
harmonization_decisions

,component,selected_variable,rationale
0,Canonical model identifier,ModelID,Globally unique and fully populated
1,Cross-dataset bridge,SangerModelID,Primary identifier shared between DepMap and GDSC
2,Primary lineage annotation,OncotreeLineage,Fully populated and suitable for lineage-aware...
3,Primary disease annotation,OncotreePrimaryDisease,Fully populated disease-level annotation
4,Subtype annotation,OncotreeSubtype,Highest available biological granularity


# Conclusions

This notebook established the identifier harmonization framework for the shared DepMap–GDSC model universe.

Key findings:

- ModelID (ACH) is fully populated and globally unique across all shared models.
- SangerModelID (SIDM) is fully populated and serves as the primary bridge between DepMap and GDSC.
- A single SIDM identifier maps to two distinct ACH identifiers, preventing SIDM from being used as a canonical identifier.
- COSMICID presents missing values and duplicate mappings and is therefore retained only as a secondary reference identifier.
- Biological annotations are fully populated across the shared universe.
- OncotreeLineage, OncotreePrimaryDisease, and OncotreeSubtype will be retained for downstream analyses.

Final harmonization decisions:

- Canonical identifier: ModelID
- Cross-dataset bridge: SangerModelID
- Primary lineage annotation: OncotreeLineage
- Primary disease annotation: OncotreePrimaryDisease
- Subtype annotation: OncotreeSubtype

These decisions define the harmonization contract that will be used during integrated cohort construction.

In [15]:
# =============================================================================
# Export harmonized model universe
# =============================================================================

harmonized_models_path = (
    INTERIM_DATA_DIR / "21_harmonized_model_universe.csv"
)

shared_depmap_models.to_csv(
    harmonized_models_path,
    index=False
)

print(f"Exported: {harmonized_models_path}")
print(f"Shape: {shared_depmap_models.shape}")

Exported: c:\Users\paula\OneDrive\Documentos\Proyectos\pancancer-epigenetics\data\interim\21_harmonized_model_universe.csv
Shape: (968, 8)
